# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = os.getenv("MLFLOW_USERNAME")
MLFLOW_PASSWORD = os.getenv("MLFLOW_PASSWORD")
EXPERIMENT_NAME = os.getenv("EXPERIMENT_NAME")

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_hamed_hamzeh
Experiment ID: 27


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [3]:
DATASET_VERSION = "v1_student"

FEATURE_DIR = Path("../hw11_etl-pipeline/data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# load the dataset.
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
    print(f"Loaded Parquet: {parquet_path}")
# raise NotImplementedError("Load feature_df from parquet_path or csv_path.")

# load metadata if metadata_path exists.
metadata = {}

if metadata_path.exists():
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

print(feature_df.shape)
feature_df.head()

Loaded Parquet: ..\hw11_etl-pipeline\data\features\listing_availability_features_v1_student.parquet
(10480, 33)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,132.0,3,356,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,89.0,2,730,...,14,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,61.0,2,730,...,16,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,730,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,1125,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [4]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# check that TARGET_COL exists.
if TARGET_COL not in feature_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found.")

# target
y = feature_df[TARGET_COL]

# clean feature list (excluding FORBIDDEN_MODEL_COLUMNS).
clean_feature_cols = [
    col for col in feature_df.columns if col not in FORBIDDEN_MODEL_COLUMNS
]

# Clean input feature.
X_clean = feature_df[clean_feature_cols]


print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64
Clean feature count: 26
['room_type', 'property_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'host_listing_count', 'neighbourhood_name', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [5]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# create leaky_feature_cols.
leaky_feature_cols = clean_feature_cols.copy()

if LEAKAGE_COLUMN not in leaky_feature_cols:
    leaky_feature_cols.append(LEAKAGE_COLUMN)

X_leaky = feature_df[leaky_feature_cols]

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 27
Leakage column included: True


In [6]:
# Normalize dtypes: pandas nullable types (Int64, boolean, etc.) → numpy types
X_clean = X_clean.astype({col: str if X_clean[col].dtype == object else float
                           for col in X_clean.columns})
X_leaky = X_leaky.astype({col: str if X_leaky[col].dtype == object else float
                           for col in X_leaky.columns})

## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [8]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# identify numeric_cols and categorical_cols from X_clean.
numeric_cols = X_clean.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = [
    col for col in X_clean.columns if col not in numeric_cols
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 23
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [21]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""

    # 1. get positive scores
    y_score = get_positive_scores(model, X_test)

    # 2. convert scores to predictions using threshold
    y_pred = (y_score >= threshold).astype(int)

    # 3. calculate accuracy, precision, recall, f1, roc_auc
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(
            y_test,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_test,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_test,
            y_pred,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_test,
            y_score,
        ),
    }

    # 4. return metrics dict, y_pred, y_score
    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [10]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # 1. create a run-specific artifact folder
    run_artifact_dir = ARTIFACT_DIR / run_name
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    # 2. save confusion_matrix.png
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )
    disp.plot(ax=ax)
    plt.tight_layout()
    cm_path = run_artifact_dir / "confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()

    # 3. save classification_report.json
    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
    report_path = (
        run_artifact_dir /
        "classification_report.json"
    )
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
        
    # 4. save feature_columns.json
    feature_path = (
        run_artifact_dir /
        "feature_columns.json"
    )
    with open(feature_path, "w") as f:
        json.dump(
            list(feature_cols),
            f,
            indent=2,
        )
    
    # 5. save dataset_metadata_snapshot.json
    metadata_path = (
        run_artifact_dir /
        "dataset_metadata_snapshot.json"
    )
    with open(metadata_path, "w") as f:
        json.dump(
            metadata,
            f,
            indent=2,
            default=str,
        )

    return run_artifact_dir



## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [ ]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    with mlflow.start_run(run_name=run_name): 
        # Fit model
        pipeline.fit(X_train, y_train)

        # Evaluate
        metrics, y_pred, y_score = evaluate_binary_classifier(
            pipeline,
            X_test,
            y_test,
            threshold=threshold,
        )

        # Save artifacts locally
        artifact_dir = save_run_artifacts(
            run_name=run_name,
            y_true=y_test,
            y_pred=y_pred,
            feature_cols=feature_cols,
            metadata=metadata,
        )

        # Log parameters
        mlflow.log_params(model_params)

        # Useful run metadata
        mlflow.log_param(
            "feature_count",
            len(feature_cols),
        )

        mlflow.log_param(
            "threshold",
            threshold,
        )

        mlflow.log_param(
            "dataset_version",
            DATASET_VERSION,
        )

        # Log metrics
        mlflow.log_metrics(metrics)

        # Log tags
        mlflow.set_tags(tags)

        # Log artifacts
        mlflow.log_artifacts(
            str(artifact_dir),
            artifact_path="artifacts",
        )

        # Log sklearn Pipeline
        # why logging this way? --> in order to handle client–server MLflow version mismatch.
        import joblib
        from pathlib import Path

        model_path = Path("pipeline.pkl")
        joblib.dump(pipeline, model_path)
        mlflow.log_artifact(str(model_path), artifact_path="model")


        run_id = mlflow.active_run().info.run_id
        print(f"Run Name : {run_name}")
        print(f"Run ID   : {run_id}")
        print("Metrics:")
        print(metrics)

        return {
            "run_id": run_id,
            "metrics": metrics,
            "pipeline": pipeline,
        }

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [13]:
# 1. split X_leaky and y using the same stratified split settings
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
# 2. build a LogisticRegression pipeline
leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=RANDOM_STATE,
                max_iter=1000,
            ),
        ),
    ]
)

leaky_params = {
    "model_type": "LogisticRegression",
    "random_state": RANDOM_STATE,
    "max_iter": 1000,
}

leaky_tags = {
    "leakage_status": "leaky",
    "known_defect": "uses future_available_rate_30d",
    "model_family": "logistic_regression",
}

# 3. log the run to MLflow
leaky_result = run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression",
    pipeline=leaky_pipeline,
    X_train=X_train_leaky,
    X_test=X_test_leaky,
    y_train=y_train_leaky,
    y_test=y_test_leaky,
    feature_cols=leaky_feature_cols,
    model_params=leaky_params,
    tags=leaky_tags,
)


Run Name : v0_leaky_logistic_regression
Run ID   : 4b56d768caf3443eae7ba4d06710a7d1
Metrics:
{'accuracy': 0.976145038167939, 'precision': 0.9760295021511985, 'recall': 0.9931207004377736, 'f1': 0.9845009299442034, 'roc_auc': 0.9880420735796894}
🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/27/runs/4b56d768caf3443eae7ba4d06710a7d1
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [15]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DummyClassifier(
                strategy="most_frequent",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

dummy_params = {
    "model_type": "DummyClassifier",
    "strategy": "most_frequent",
}

dummy_tags = {
    "leakage_status": "clean",
    "model_family": "dummy_baseline",
}

dummy_result = run_mlflow_experiment(
    run_name="v1_dummy_baseline",
    pipeline=dummy_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=dummy_params,
    tags=dummy_tags,
)


Run Name : v1_dummy_baseline
Run ID   : 77f97df1b3f64637a6938e017955765d
Metrics:
{'accuracy': 0.762881679389313, 'precision': 0.762881679389313, 'recall': 1.0, 'f1': 0.8654939106901218, 'roc_auc': 0.5}
🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/27/runs/77f97df1b3f64637a6938e017955765d
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [ ]:
clean_lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=RANDOM_STATE,
                max_iter=1000,
            ),
        ),
    ]
)

clean_lr_params = {
    "model_type": "LogisticRegression",
    "max_iter": 1000,
    "random_state": RANDOM_STATE,
}

clean_lr_tags = {
    "leakage_status": "clean",
    "model_family": "logistic_regression",
}

clean_lr_result = run_mlflow_experiment(
    run_name="v2_clean_logistic_regression",
    pipeline=clean_lr_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=clean_lr_params,
    tags=clean_lr_tags,
)


Run Name : v2_clean_logistic_regression
Run ID   : f578e4b3c730407eb0d575e7a43cae00
Metrics:
{'accuracy': 0.976145038167939, 'precision': 0.9760295021511985, 'recall': 0.9931207004377736, 'f1': 0.9845009299442034, 'roc_auc': 0.9880420735796894}
🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/27/runs/f578e4b3c730407eb0d575e7a43cae00
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [18]:
balanced_lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=RANDOM_STATE,
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)

balanced_lr_params = {
    "model_type": "LogisticRegression",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
}

balanced_lr_tags = {
    "leakage_status": "clean",
    "model_family": "logistic_regression",
}

balanced_lr_result = run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression",
    pipeline=balanced_lr_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=balanced_lr_params,
    tags=balanced_lr_tags,
)


Run Name : v3_balanced_logistic_regression
Run ID   : fa42bd8b7f6d4336aeb63adad1227b9c
Metrics:
{'accuracy': 0.9770992366412213, 'precision': 0.9819763828464885, 'recall': 0.9881175734834271, 'f1': 0.9850374064837906, 'roc_auc': 0.9894853800728071}
🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/27/runs/fa42bd8b7f6d4336aeb63adad1227b9c
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [22]:
thresholds = [0.3, 0.40, 0.50, 0.60]

for t in thresholds:

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    random_state=RANDOM_STATE,
                    max_iter=1000,
                ),
            ),
        ]
    )

    params = {
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "decision_threshold": t,
    }

    tags = {
        "leakage_status": "clean",
        "model_family": "logistic_regression",
        "experiment_type": "threshold_tuning",
    }

    run_mlflow_experiment(
        run_name=f"v4_threshold_{t}",
        pipeline=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params=params,
        tags=tags,
        threshold=t,
    )


Run Name : v4_threshold_0.3
Run ID   : dab3cf1cff0745c7aba80dd84198a1be
Metrics:
{'accuracy': 0.9737595419847328, 'precision': 0.9724602203182374, 'recall': 0.9937460913070669, 'f1': 0.9829879369007114, 'roc_auc': 0.9880420735796894}
🏃 View run v4_threshold_0.3 at: http://185.50.38.163:33014/#/experiments/27/runs/dab3cf1cff0745c7aba80dd84198a1be
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27
Run Name : v4_threshold_0.4
Run ID   : afbffef158684ffcb6d3dfed74b2c791
Metrics:
{'accuracy': 0.9742366412213741, 'precision': 0.9736358062538321, 'recall': 0.9931207004377736, 'f1': 0.98328173374613, 'roc_auc': 0.9880420735796894}
🏃 View run v4_threshold_0.4 at: http://185.50.38.163:33014/#/experiments/27/runs/afbffef158684ffcb6d3dfed74b2c791
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27
Run Name : v4_threshold_0.5
Run ID   : 8d89af8d3e9c4f6abcfe656482150f06
Metrics:
{'accuracy': 0.976145038167939, 'precision': 0.9760295021511985, 'recall': 0.9931207004377736

## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [25]:
n_estimators=200
max_depth=10
min_samples_leaf=2
class_weight="balanced"

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                class_weight=class_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_params = {
    "model_type": "RandomForestClassifier",
    "n_estimators": n_estimators,
    "max_depth": max_depth,
    "min_samples_leaf": min_samples_leaf,
    "class_weight": class_weight,
    "random_state": RANDOM_STATE,
}

rf_tags = {
    "leakage_status": "clean",
    "model_family": "random_forest",
}

rf_result = run_mlflow_experiment(
    run_name="v5_random_forest",
    pipeline=rf_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params=rf_params,
    tags=rf_tags,
)

Run Name : v5_random_forest
Run ID   : ae07c048c3e1432581c322f90b0275f0
Metrics:
{'accuracy': 0.982824427480916, 'precision': 0.9887429643527205, 'recall': 0.9887429643527205, 'f1': 0.9887429643527205, 'roc_auc': 0.9946785151182267}
🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/27/runs/ae07c048c3e1432581c322f90b0275f0
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/27


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [26]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

comparison_df = runs[
    [
        "tags.mlflow.runName",
        "tags.leakage_status",
        "tags.model_family",
        "metrics.accuracy",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
    ]
].sort_values("metrics.f1", ascending=False)

comparison_df

,tags.mlflow.runName,tags.leakage_status,tags.model_family,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc
0,v5_random_forest,clean,random_forest,0.982824,0.988743,0.988743,0.988743,0.994679
5,v3_balanced_logistic_regression,clean,logistic_regression,0.977099,0.981976,0.988118,0.985037,0.989485
1,v4_threshold_0.6,clean,logistic_regression,0.976622,0.976630,0.993121,0.984806,0.988042
6,v2_clean_logistic_regression,clean,logistic_regression,0.976145,0.976030,0.993121,0.984501,0.988042
2,v4_threshold_0.5,clean,logistic_regression,0.976145,0.976030,0.993121,0.984501,0.988042
8,v0_leaky_logistic_regression,leaky,logistic_regression,0.976145,0.976030,0.993121,0.984501,0.988042
3,v4_threshold_0.4,clean,logistic_regression,0.974237,0.973636,0.993121,0.983282,0.988042
4,v4_threshold_0.3,clean,logistic_regression,0.973760,0.972460,0.993746,0.982988,0.988042
7,v1_dummy_baseline,clean,dummy_baseline,0.762882,0.762882,1.000000,0.865494,0.500000


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [ ]:
# set BEST_RUN_ID to the selected clean run ID.
# random forest model got selected because of highest f1 score in comparison with other models.
BEST_RUN_ID = "ae07c048c3e1432581c322f90b0275f0"

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

Selected best run: ae07c048c3e1432581c322f90b0275f0


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [30]:
final_explanation = """
Several models were trained and tracked using MLflow, including a dummy baseline, multiple logistic regression variants, threshold-tuned models, and a random forest classifier.
The dummy baseline served as a reference point and achieved an accuracy of approximately 0.76 with a ROC-AUC of 0.50, indicating no meaningful predictive capability beyond always predicting the majority class.
The logistic regression models performed significantly better, with F1 scores around 0.98. Adjusting class weights slightly improved recall for the minority class,
and threshold tuning allowed small adjustments to the precision–recall trade-off.

However, the Random Forest model (v5_random_forest) achieved the best overall performance across all key metrics:
Accuracy: 0.9828
Precision: 0.9887
Recall: 0.9887
F1 Score: 0.9887
ROC-AUC: 0.9947

This model provides the strongest balance between precision and recall and shows the highest discriminative ability according to ROC-AUC.
The earlier leaky logistic regression model was excluded from consideration because it relied on features that introduced data leakage.
Such models may appear to perform well but would fail in real-world deployment since they use information not available at prediction time.
Based on the comparison of all valid experiments, the Random Forest model was selected as the production candidate and tagged accordingly in MLflow.
"""

print(final_explanation)


Several models were trained and tracked using MLflow, including a dummy baseline, multiple logistic regression variants, threshold-tuned models, and a random forest classifier.
The dummy baseline served as a reference point and achieved an accuracy of approximately 0.76 with a ROC-AUC of 0.50, indicating no meaningful predictive capability beyond always predicting the majority class.
The logistic regression models performed significantly better, with F1 scores around 0.98. Adjusting class weights slightly improved recall for the minority class,
and threshold tuning allowed small adjustments to the precision–recall trade-off.

However, the Random Forest model (v5_random_forest) achieved the best overall performance across all key metrics:
Accuracy: 0.9828
Precision: 0.9887
Recall: 0.9887
F1 Score: 0.9887
ROC-AUC: 0.9947

This model provides the strongest balance between precision and recall and shows the highest discriminative ability according to ROC-AUC.
The earlier leaky logistic re